[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/08-ai-engineering/07_fine-tuning.ipynb)

*AI Engineering · Notebook 7 of 10*

# Fine-Tuning in Practice: LoRA, QLoRA & DPO

Adapt open models yourself: when to fine-tune, LoRA maths and merging, memory for full vs LoRA vs QLoRA, chat templates and loss masking, packing, cleaning and de-duplicating data, schedules and batch sizes, preference data and the DPO loss, early stopping and checking for forgetting.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** LoRA is sticky notes on a textbook: the book stays the same, the notes change how you read it, and you can swap notes per student.

**What interviewers really test here:** When to fine-tune vs prompt vs RAG, LoRA maths and rank, QLoRA memory, loss masking, DPO, leakage and forgetting.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. When fine-tuning is the right tool *(Basics)*
- [ ] 2. LoRA: train a tiny update instead of the whole model *(Core)*
- [ ] 3. Memory: full vs LoRA vs QLoRA *(Core)*
- [ ] 4. Data and training: where results are won *(Core)*
- [ ] 5. Preference tuning with DPO *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [ ]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import math
import json
import numpy as np
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Kinds of fine-tuning

| Method | Data | Teaches |
| --- | --- | --- |
| Continued pre-training | raw domain text | domain language (expensive) |
| SFT (instruction tuning) | prompt → ideal response | format, style, task skill |
| Preference tuning (DPO, RLHF) | prompt + chosen + rejected | which answers people prefer |
| RL with verifiable rewards | problems with checkable answers | reasoning on maths / code |
| Distillation | teacher model outputs | a small model imitating a big one |

### Parameter-efficient methods

- **LoRA**: low-rank update A·B per chosen matrix; rank r and scale α are the key knobs.
- **QLoRA**: LoRA on a 4-bit quantised frozen base (NF4), with bf16 compute.
- Variants (DoRA, LoRA+, rank-stabilised scaling) tweak the same idea.
- Adapters can be merged into W for serving, or hot-swapped per tenant.

---
# Part 2: Lessons, step by step

## Lesson 1: When fine-tuning is the right tool

*Level: Basics*

Fine-tuning changes a model's weights with your examples. Try cheaper options first:



| Problem | Best first tool |
| --- | --- |
| Model doesn't know your facts / documents | **RAG** — facts change; weights shouldn't |
| Output format or tone is off | **Prompting** with examples, then fine-tune if you have hundreds+ of examples |
| A narrow task done millions of times | **Fine-tune a small model** (often by distilling a big one) for cost and latency |
| Model should prefer some answers over others | **Preference tuning** (DPO) |



Fine-tuning teaches **behaviour** (format, style, a skill), not reliable new **knowledge**. And always compare against a well-prompted baseline on your eval set.

**✏️ Your turn:** you now have what you need for [Q20](#q20), [Q8](#q8). Try one now, or carry on and do them in Part 3.

## Lesson 2: LoRA: train a tiny update instead of the whole model

*Level: Core*

Full fine-tuning updates every weight — billions of numbers, with gradients and optimizer state for each. **LoRA** (Low-Rank Adaptation) freezes the original weight W and learns a small update as the product of two thin matrices:



```python
h = x @ W + (alpha / r) * (x @ A) @ B      # W: d×k frozen;  A: d×r,  B: r×k trained;  r ≪ d, k
```



With r = 16 on a 4096×4096 matrix that's 0.8% of the parameters. B starts at zero, so training begins exactly at the original model. After training you can **merge** A·B into W (no extra serving cost), or keep adapters separate and swap them per customer.



```python
from peft import LoraConfig, get_peft_model
config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])
model = get_peft_model(base_model, config)
model.print_trainable_parameters()
```

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q10](#q10), [Q11](#q11), [Q12](#q12), [Q2](#q2), [Q9](#q9). Try one now, or carry on and do them in Part 3.

## Lesson 3: Memory: full vs LoRA vs QLoRA

*Level: Core*

Training memory per parameter, roughly: full fine-tuning with Adam needs ~16 bytes (weights, gradients, two optimizer moments, master copy) — about 128 GB for an 8B model before activations. LoRA keeps the frozen weights in bf16 (2 bytes) and pays the 16 bytes only for the small adapter. **QLoRA** stores the frozen weights in 4 bits (0.5 bytes), which puts 7–8B models on a single consumer GPU or a free Colab GPU.



```python
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype="bfloat16")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-7B-Instruct", quantization_config=bnb)
```

**✏️ Your turn:** you now have what you need for [Q13](#q13). Try one now, or carry on and do them in Part 3.

## Lesson 4: Data and training: where results are won

*Level: Core*

- **Format** every example with the model's own chat template, and compute the loss only on the assistant's tokens.
- **Quality over quantity**: a few thousand clean, diverse, correct examples beat a hundred thousand noisy ones. Remove empties, over-long items and duplicates, and make sure no eval prompts leak into training.
- **Packing** short examples into full sequences wastes less compute on padding.
- **Hyper-parameters**: LoRA learning rates around 1e-4 to 2e-4 with warm-up and cosine decay; a few epochs; watch validation loss and stop early when it rises.



```python
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(model=model, train_dataset=train, eval_dataset=val, peft_config=config,
                     args=SFTConfig(learning_rate=2e-4, num_train_epochs=2, per_device_train_batch_size=4,
                                    gradient_accumulation_steps=8, lr_scheduler_type="cosine", warmup_ratio=0.05))
trainer.train()
```



Library arguments change between versions — check the docs for the versions you install.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [ ]:
import matplotlib.pyplot as plt


def schedule_plot():
    steps = np.arange(0, 1001)
    lr = np.where(steps < 100, 2e-4 * steps / 100, 2e-4 * 0.5 * (1 + np.cos(np.pi * (steps - 100) / 900)))
    fig, ax = plt.subplots(figsize=(5, 2.4))
    ax.plot(steps, lr * 1e6)
    ax.set_xlabel("step"); ax.set_ylabel("learning rate (×1e-6)"); ax.set_title("Warm-up, then cosine decay")


schedule_plot()

**✏️ Your turn:** you now have what you need for [Q3](#q3), [Q14](#q14), [Q15](#q15), [Q16](#q16), [Q4](#q4), [Q5](#q5), [Q6](#q6). Try one now, or carry on and do them in Part 3.

## Lesson 5: Preference tuning with DPO

*Level: Advanced*

Supervised fine-tuning shows the model good answers. **Preference tuning** shows pairs — a chosen and a rejected answer to the same prompt — and teaches it to prefer the chosen kind. **DPO** (Direct Preference Optimisation) does this with a simple classification-style loss, no separate reward model or RL loop:



```python
margin = beta * ((logp_chosen - ref_chosen) - (logp_rejected - ref_rejected))
loss = -log(sigmoid(margin))          # push the margin up, relative to a frozen reference copy
```



β controls how far the model may drift from the reference. Watch **reward accuracy** (share of pairs with a positive margin) during training. Build pairs from ratings or A/B judgements, keeping only clear gaps.



After any fine-tune, re-run general benchmarks: narrow training can erode other skills (**catastrophic forgetting**) and safety behaviour.

**✏️ Your turn:** you now have what you need for [Q17](#q17), [Q18](#q18), [Q19](#q19), [Q7](#q7). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

<a id="q1"></a>
### Q1. How Small Is a LoRA Adapter?

🟢 Warm-up · *Parameter counts*

LoRA replaces training a d×k matrix with two small matrices A (d×r) and B (r×k). Return `(full params, LoRA params, LoRA as % of full rounded to 3)`.

**Examples**

- `d = 4096, k = 4096, r = 16` → `(16777216, 131072, 0.781)`
- `d = 4096, k = 14336, r = 8` → `(58720256, 147456, 0.251)`

In [ ]:
def lora_params(d, k, r):
    """
    LoRA replaces training a d×k matrix with two small matrices A (d×r) and B
    (r×k). Return (full params, LoRA params, LoRA as % of full rounded to 3).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(lora_params, [
    ((4096, 4096, 16), (16777216, 131072, 0.781)),
    ((4096, 14336, 8), (58720256, 147456, 0.251)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Multiply step by step.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

One tuple expression.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Multiply step by step.

```python
def lora_params(d, k, r):
    full = d * k
    lora = d * r + r * k
    return (full, lora, round(lora / full * 100, 3))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** One tuple expression.

```python
def lora_params(d, k, r):
    return (d * k, r * (d + k), round(r * (d + k) / (d * k) * 100, 3))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Multiply step by step. That costs O(1) time and O(1) space."

"I can do better: One tuple expression. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/fine-tuning/how-small-is-a-lora-adapter)

<a id="q2"></a>
### Q2. Why B Starts at Zero

🟢 Warm-up · *Safe initialisation*

LoRA initialises A randomly and B to zeros, so at step 0 the adapted model behaves exactly like the original. Return `(max |LoRA output − base output| with B = 0, the same with B random)`, rounded to 6.

**Examples**

- `x = [[0.13, -0.13, 0.64, 0.1], [-0.54, 0.36, 1.3, 0.95]], W = [[-0.7, -1.27, -0.62], [0.04, -2.33, -0.22], [-1.25, -0.73, -0.54], [-0.32, 0.41, 1.04]], A = [[-0.013, 0.137], [-0.067, 0.035], [0.09, 0.009], [-0.074, -0.092]], B = [[-0.046, 0.022, -0.101], [-0.021, -0.016, 0.054]], alpha = 16` → `(0.0, 0.0831)`

In [ ]:
def zero_init(x, W, A, B, alpha):
    """
    LoRA initialises A randomly and B to zeros, so at step 0 the adapted model
    behaves exactly like the original. Return (max |LoRA output − base output|
    with B = 0, the same with B random), rounded to 6.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(zero_init, [
    (([[0.13, -0.13, 0.64, 0.1], [-0.54, 0.36, 1.3, 0.95]], [[-0.7, -1.27, -0.62], [0.04, -2.33, -0.22], [-1.25, -0.73, -0.54], [-0.32, 0.41, 1.04]], [[-0.013, 0.137], [-0.067, 0.035], [0.09, 0.009], [-0.074, -0.092]], [[-0.046, 0.022, -0.101], [-0.021, -0.016, 0.054]], 16), (0.0, 0.0831)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute both outputs with loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

NumPy, with `np.zeros_like(B)`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Compute both outputs with loops.

```python
def zero_init(x, W, A, B, alpha):
    x, W, A, B = map(np.array, (x, W, A, B))
    r = A.shape[1]
    base = x @ W
    zero_B = [[0.0] * B.shape[1] for _ in range(B.shape[0])]
    with_zero = base + alpha / r * (x @ A) @ np.array(zero_B)
    with_rand = base + alpha / r * (x @ A) @ B
    return (round(float(abs(with_zero - base).max()), 6), round(float(abs(with_rand - base).max()), 6))
```

**Cost:** O(n·d·k) time, O(n·k) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** NumPy, with `np.zeros_like(B)`.

```python
def zero_init(x, W, A, B, alpha):
    x, W, A, B = map(np.array, (x, W, A, B))
    r = A.shape[1]
    out = lambda Bm: x @ W + alpha / r * (x @ A) @ Bm
    return (round(float(np.abs(out(np.zeros_like(B)) - x @ W).max()), 6), round(float(np.abs(out(B) - x @ W).max()), 6))
```

**Cost:** O(n·d·k) time, O(n·k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute both outputs with loops. That costs O(n·d·k) time and O(n·k) space."

"I can do better: NumPy, with `np.zeros_like(B)`. That's O(n·d·k) time and O(n·k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/fine-tuning/why-b-starts-at-zero)

<a id="q3"></a>
### Q3. Apply a Chat Template

🟢 Warm-up · *Formatting training data*

Fine-tuning data must use the model's own chat format. Render messages as `<|role|>\ncontent<|end|>\n` for each, and end with `<|assistant|>\n` if `add_generation_prompt`. (Real models each have their own template; `tokenizer.apply_chat_template` applies it.)

**Examples**

- `messages = [{'role': 'system', 'content': 'Be brief.'}, {'role': 'user', 'content': 'Hi'}, {'role': 'assistant', 'content': 'Hello!'}], add_generation_prompt = False` → `'<|system|>\nBe brief.<|end|>\n<|user|>\nHi<|end|>\n<|assistant|>\nHello!<|end|>\n'`
- `messages = [{'role': 'user', 'content': '2+2?'}], add_generation_prompt = True` → `'<|user|>\n2+2?<|end|>\n<|assistant|>\n'`

In [ ]:
def chat_template(messages, add_generation_prompt):
    """
    Fine-tuning data must use the model's own chat format. Render messages as
    <|role|>\ncontent<|end|>\n for each, and end with <|assistant|>\n if
    add_generation_prompt. (Real models each have their own template;
    tokenizer.apply_chat_template applies it.)
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(chat_template, [
    (([{'role': 'system', 'content': 'Be brief.'}, {'role': 'user', 'content': 'Hi'}, {'role': 'assistant', 'content': 'Hello!'}], False), '<|system|>\nBe brief.<|end|>\n<|user|>\nHi<|end|>\n<|assistant|>\nHello!<|end|>\n'),
    (([{'role': 'user', 'content': '2+2?'}], True), '<|user|>\n2+2?<|end|>\n<|assistant|>\n'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

String concatenation in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`''.join` over f-strings.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** String concatenation in a loop.

```python
def chat_template(messages, add_generation_prompt):
    text = ""
    for m in messages:
        text += "<|" + m["role"] + "|>\n" + m["content"] + "<|end|>\n"
    if add_generation_prompt:
        text += "<|assistant|>\n"
    return text
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `''.join` over f-strings.

```python
def chat_template(messages, add_generation_prompt):
    return "".join(f"<|{m['role']}|>\n{m['content']}<|end|>\n" for m in messages) + ("<|assistant|>\n" if add_generation_prompt else "")
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: String concatenation in a loop. That costs O(n) time and O(n) space."

"I can do better: `''.join` over f-strings. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/fine-tuning/apply-a-chat-template)

<a id="q4"></a>
### Q4. Warm-Up Then Cosine

🟢 Warm-up · *Learning-rate schedule*

The learning rate rises linearly from 0 to `peak` over `warmup` steps, then follows a cosine down to 0 at `total` steps. Return the rate at each step in `steps` in **millionths** (rate × 10⁶), rounded to 2.

**Examples**

- `peak = 0.0002, warmup = 100, total = 1000, steps = [0, 50, 100, 550, 1000]` → `[0.0, 100.0, 200.0, 100.0, 0.0]`

In [ ]:
def lr_schedule(peak, warmup, total, steps):
    """
    The learning rate rises linearly from 0 to peak over warmup steps, then
    follows a cosine down to 0 at total steps. Return the rate at each step in
    steps in millionths (rate × 10⁶), rounded to 2.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(lr_schedule, [
    ((0.0002, 100, 1000, [0, 50, 100, 550, 1000]), [0.0, 100.0, 200.0, 100.0, 0.0]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

An if/else per step.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A helper function and a comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** An if/else per step.

```python
def lr_schedule(peak, warmup, total, steps):
    out = []
    for s in steps:
        if s < warmup:
            lr = peak * s / warmup
        else:
            progress = (s - warmup) / (total - warmup)
            lr = peak * 0.5 * (1 + math.cos(math.pi * progress))
        out.append(round(lr * 1e6, 2))
    return out
```

**Cost:** O(s) time, O(s) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A helper function and a comprehension.

```python
def lr_schedule(peak, warmup, total, steps):
    lr = lambda s: peak * s / warmup if s < warmup else peak * 0.5 * (1 + math.cos(math.pi * (s - warmup) / (total - warmup)))
    return [round(lr(s) * 1e6, 2) for s in steps]
```

**Cost:** O(s) time, O(s) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: An if/else per step. That costs O(s) time and O(s) space."

"I can do better: A helper function and a comprehension. That's O(s) time and O(s) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/fine-tuning/warm-up-then-cosine)

<a id="q5"></a>
### Q5. Batch Size and Steps

🟢 Warm-up · *Gradient accumulation*

Effective batch = per-device batch × accumulation steps × GPUs. Return `(effective batch, optimizer steps per epoch rounded up, total steps for `epochs`)` for `n` examples.

**Examples**

- `per_device = 4, accum = 8, gpus = 2, n = 10000, epochs = 3` → `(64, 157, 471)`
- `per_device = 1, accum = 16, gpus = 1, n = 1200, epochs = 2` → `(16, 75, 150)`

In [ ]:
def batch_steps(per_device, accum, gpus, n, epochs):
    """
    Effective batch = per-device batch × accumulation steps × GPUs. Return
    (effective batch, optimizer steps per epoch rounded up, total steps for
    epochs) for n examples.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(batch_steps, [
    ((4, 8, 2, 10000, 3), (64, 157, 471)),
    ((1, 16, 1, 1200, 2), (16, 75, 150)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Multiply, then ceiling divide.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`math.ceil`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Multiply, then ceiling divide.

```python
def batch_steps(per_device, accum, gpus, n, epochs):
    eff = per_device * accum * gpus
    per_epoch = n // eff
    if n % eff:
        per_epoch += 1
    return (eff, per_epoch, per_epoch * epochs)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `math.ceil`.

```python
def batch_steps(per_device, accum, gpus, n, epochs):
    eff = per_device * accum * gpus
    return (eff, math.ceil(n / eff), math.ceil(n / eff) * epochs)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Multiply, then ceiling divide. That costs O(1) time and O(1) space."

"I can do better: `math.ceil`. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/fine-tuning/batch-size-and-steps)

<a id="q6"></a>
### Q6. Stop Before It Overfits

🟢 Warm-up · *Early stopping*

Validation loss per epoch is given. Stop once it hasn't improved for `patience` epochs. Return `(best epoch, its loss, epoch at which training stops)` (epochs counted from 1).

**Examples**

- `val_losses = [1.9, 1.5, 1.32, 1.28, 1.3, 1.35, 1.41, 1.5], patience = 2` → `(4, 1.28, 6)`
- `val_losses = [2.0, 1.8, 1.6], patience = 2` → `(3, 1.6, 3)`

In [ ]:
def early_stop(val_losses, patience):
    """
    Validation loss per epoch is given. Stop once it hasn't improved for
    patience epochs. Return (best epoch, its loss, epoch at which training
    stops) (epochs counted from 1).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(early_stop, [
    (([1.9, 1.5, 1.32, 1.28, 1.3, 1.35, 1.41, 1.5], 2), (4, 1.28, 6)),
    (([2.0, 1.8, 1.6], 2), (3, 1.6, 3)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop tracking best and a counter.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same loop with `enumerate(…, 1)`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A loop tracking best and a counter.

```python
def early_stop(val_losses, patience):
    best, best_epoch, waited = float("inf"), 0, 0
    stop = len(val_losses)
    for i in range(len(val_losses)):
        if val_losses[i] < best:
            best, best_epoch, waited = val_losses[i], i + 1, 0
        else:
            waited += 1
            if waited >= patience:
                stop = i + 1
                break
    return (best_epoch, best, stop)
```

**Cost:** O(e) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same loop with `enumerate(…, 1)`.

```python
def early_stop(val_losses, patience):
    best, best_epoch = float("inf"), 0
    for epoch, loss in enumerate(val_losses, 1):
        if loss < best:
            best, best_epoch = loss, epoch
        elif epoch - best_epoch >= patience:
            return (best_epoch, best, epoch)
    return (best_epoch, best, len(val_losses))
```

**Cost:** O(e) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop tracking best and a counter. That costs O(e) time and O(1) space."

"I can do better: Same loop with `enumerate(…, 1)`. That's O(e) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/fine-tuning/stop-before-it-overfits)

<a id="q7"></a>
### Q7. Did It Forget Everything Else?

🟢 Warm-up · *Catastrophic forgetting*

Compare benchmark scores before and after fine-tuning. Return `(target task gain rounded to 1, general benchmarks that dropped by more than `tol` points)`.

**Examples**

- `before = {'target': 61.0, 'reasoning': 72.0, 'coding': 55.0, 'safety': 93.0}, after = {'target': 84.5, 'reasoning': 70.5, 'coding': 47.0, 'safety': 86.0}, tol = 2.0` → `(23.5, ['coding', 'safety'])`

In [ ]:
def forgetting(before, after, tol):
    """
    Compare benchmark scores before and after fine-tuning. Return (target task
    gain rounded to 1, general benchmarks that dropped by more than tol points).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(forgetting, [
    (({'target': 61.0, 'reasoning': 72.0, 'coding': 55.0, 'safety': 93.0}, {'target': 84.5, 'reasoning': 70.5, 'coding': 47.0, 'safety': 86.0}, 2.0), (23.5, ['coding', 'safety'])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over benchmarks.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop over benchmarks.

```python
def forgetting(before, after, tol):
    dropped = []
    for b in before:
        if b != "target" and before[b] - after[b] > tol:
            dropped.append(b)
    return (round(after["target"] - before["target"], 1), dropped)
```

**Cost:** O(b) time, O(b) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A comprehension.

```python
def forgetting(before, after, tol):
    return (round(after["target"] - before["target"], 1), [b for b in before if b != "target" and before[b] - after[b] > tol])
```

**Cost:** O(b) time, O(b) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over benchmarks. That costs O(b) time and O(b) space."

"I can do better: A comprehension. That's O(b) time and O(b) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/fine-tuning/did-it-forget-everything-else)

<a id="q8"></a>
### Q8. What Does a Distillation Dataset Cost?

🟢 Warm-up · *Synthetic data*

Generate `n` training examples with a large teacher model: each call reads `inp` tokens and writes `out` tokens, at prices per million tokens. A batch API halves the price. Return `(cost, cost with batch discount)` in dollars rounded to 2.

**Examples**

- `n = 20000, inp = 600, out = 400, pin = 3.0, pout = 15.0` → `(156.0, 78.0)`

In [ ]:
def distil_cost(n, inp, out, pin, pout):
    """
    Generate n training examples with a large teacher model: each call reads inp
    tokens and writes out tokens, at prices per million tokens. A batch API
    halves the price. Return (cost, cost with batch discount) in dollars rounded
    to 2.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(distil_cost, [
    ((20000, 600, 400, 3.0, 15.0), (156.0, 78.0)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Multiply step by step.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

One expression.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Multiply step by step.

```python
def distil_cost(n, inp, out, pin, pout):
    per = inp * pin / 1e6 + out * pout / 1e6
    total = per * n
    return (round(total, 2), round(total / 2, 2))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** One expression.

```python
def distil_cost(n, inp, out, pin, pout):
    total = n * (inp * pin + out * pout) / 1e6
    return (round(total, 2), round(total * 0.5, 2))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Multiply step by step. That costs O(1) time and O(1) space."

"I can do better: One expression. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/fine-tuning/what-does-a-distillation-dataset-cost)

<a id="q9"></a>
### Q9. Choose the LoRA Rank

🟢 Warm-up · *Good enough, smallest*

`RANK_EVAL[r] = (eval score, adapter size in MB)`. Return the smallest rank whose score is within `tol` points of the best score.

**Examples**

- `tol = 1.0` → `16`
- `tol = 0.2` → `32`
- `tol = 5.0` → `8`

In [ ]:
RANK_EVAL = {4: (71.2, 4.2), 8: (74.8, 8.4), 16: (76.1, 16.8), 32: (76.4, 33.6), 64: (76.5, 67.1)}

def pick_rank(tol):
    """
    RANK_EVAL[r] = (eval score, adapter size in MB). Return the smallest rank
    whose score is within tol points of the best score.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(pick_rank, [
    ((1.0,), 16),
    ((0.2,), 32),
    ((5.0,), 8),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Find the best, then loop over ranks in order.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`min` over a filtered generator.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Find the best, then loop over ranks in order.

```python
def pick_rank(tol):
    best = max(score for score, _ in RANK_EVAL.values())
    for r in sorted(RANK_EVAL):
        if best - RANK_EVAL[r][0] <= tol:
            return r
```

**Cost:** O(r) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `min` over a filtered generator.

```python
def pick_rank(tol):
    best = max(s for s, _ in RANK_EVAL.values())
    return min(r for r, (s, _) in RANK_EVAL.items() if best - s <= tol + 1e-9)
```

**Cost:** O(r) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Find the best, then loop over ranks in order. That costs O(r) time and O(1) space."

"I can do better: `min` over a filtered generator. That's O(r) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/fine-tuning/choose-the-lora-rank)

<a id="q10"></a>
### Q10. Trainable Share of a Whole Model

🟡 Interview standard · *Choosing target modules*

A model has `layers` layers; in each, LoRA adapts the matrices in `shapes` (name → (d, k)). Return `(trainable LoRA params, % of the model's `total_b` billion params, rounded to 3)`.

**Examples**

- `layers = 32, shapes = {'q_proj': (4096, 4096), 'k_proj': (4096, 1024), 'v_proj': (4096, 1024), 'o_proj': (4096, 4096)}, r = 16, total_b = 8.0` → `(13631488, 0.17)`
- `layers = 32, shapes = {'q_proj': (4096, 4096), 'v_proj': (4096, 1024)}, r = 8, total_b = 8.0` → `(3407872, 0.043)`

In [ ]:
def model_lora(layers, shapes, r, total_b):
    """
    A model has layers layers; in each, LoRA adapts the matrices in shapes (name
    → (d, k)). Return (trainable LoRA params, % of the model's total_b billion
    params, rounded to 3).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(model_lora, [
    ((32, {'q_proj': (4096, 4096), 'k_proj': (4096, 1024), 'v_proj': (4096, 1024), 'o_proj': (4096, 4096)}, 16, 8.0), (13631488, 0.17)),
    ((32, {'q_proj': (4096, 4096), 'v_proj': (4096, 1024)}, 8, 8.0), (3407872, 0.043)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Nested loop over layers and matrices.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Per-layer sum times the number of layers.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Nested loop over layers and matrices.

```python
def model_lora(layers, shapes, r, total_b):
    n = 0
    for _ in range(layers):
        for d, k in shapes.values():
            n += r * (d + k)
    return (n, round(n / (total_b * 1e9) * 100, 3))
```

**Cost:** O(L · m) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Per-layer sum times the number of layers.

```python
def model_lora(layers, shapes, r, total_b):
    n = layers * sum(r * (d + k) for d, k in shapes.values())
    return (n, round(n / (total_b * 1e9) * 100, 3))
```

**Cost:** O(m) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Nested loop over layers and matrices. That costs O(L · m) time and O(1) space."

"I can do better: Per-layer sum times the number of layers. That's O(m) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/fine-tuning/trainable-share-of-a-whole-model)

<a id="q11"></a>
### Q11. The LoRA Forward Pass

🟡 Interview standard · *x W + (α/r) x A B*

Compute the LoRA layer output h = x·W + (alpha / r)·(x·A)·B, rounded to 4. r is the number of columns of A.

**Examples**

- `x = [[0.13, -0.13, 0.64, 0.1], [-0.54, 0.36, 1.3, 0.95]], W = [[-0.7, -1.27, -0.62], [0.04, -2.33, -0.22], [-1.25, -0.73, -0.54], [-0.32, 0.41, 1.04]], A = [[-0.013, 0.137], [-0.067, 0.035], [0.09, 0.009], [-0.074, -0.092]], B = [[-0.046, 0.022, -0.101], [-0.021, -0.016, 0.054]], alpha = 16` → `[[-0.9509, -0.2796, -0.3356], [-1.5245, -0.6897, 0.4585]]`

In [ ]:
def lora_forward(x, W, A, B, alpha):
    """
    Compute the LoRA layer output h = x·W + (alpha / r)·(x·A)·B, rounded to 4. r
    is the number of columns of A.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(lora_forward, [
    (([[0.13, -0.13, 0.64, 0.1], [-0.54, 0.36, 1.3, 0.95]], [[-0.7, -1.27, -0.62], [0.04, -2.33, -0.22], [-1.25, -0.73, -0.54], [-0.32, 0.41, 1.04]], [[-0.013, 0.137], [-0.067, 0.035], [0.09, 0.009], [-0.074, -0.092]], [[-0.046, 0.022, -0.101], [-0.021, -0.016, 0.054]], 16), [[-0.9509, -0.2796, -0.3356], [-1.5245, -0.6897, 0.4585]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Matrix multiplication with nested loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

NumPy `@`. Note it never forms the full d×k update.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Matrix multiplication with nested loops.

```python
def lora_forward(x, W, A, B, alpha):
    def matmul(P, Q):
        return [[sum(P[i][t] * Q[t][j] for t in range(len(Q))) for j in range(len(Q[0]))] for i in range(len(P))]
    r = len(A[0])
    base = matmul(x, W)
    delta = matmul(matmul(x, A), B)
    return [[round(base[i][j] + alpha / r * delta[i][j], 4) for j in range(len(base[0]))] for i in range(len(base))]
```

**Cost:** O(n·d·k) time, O(n·k) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** NumPy `@`. Note it never forms the full d×k update.

```python
def lora_forward(x, W, A, B, alpha):
    x, W, A, B = map(np.array, (x, W, A, B))
    return np.round(x @ W + alpha / A.shape[1] * (x @ A) @ B, 4).tolist()
```

**Cost:** O(n·d·(k + r)) time, O(n·k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Matrix multiplication with nested loops. That costs O(n·d·k) time and O(n·k) space."

"I can do better: NumPy `@`. Note it never forms the full d×k update. That's O(n·d·(k + r)) time and O(n·k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/fine-tuning/the-lora-forward-pass)

<a id="q12"></a>
### Q12. Merge the Adapter for Serving

🟡 Interview standard · *W' = W + (α/r)·A·B*

After training, LoRA can be folded into the weights so serving has no extra cost. Return `(merged W rounded to 4, max |difference| between x·W' and the LoRA forward, rounded to 8)`.

**Examples**

- `x = [[0.13, -0.13, 0.64, 0.1], [-0.54, 0.36, 1.3, 0.95]], W = [[-0.7, -1.27, -0.62], [0.04, -2.33, -0.22], [-1.25, -0.73, -0.54], [-0.32, 0.41, 1.04]], A = [[-0.013, 0.137], [-0.067, 0.035], [0.09, 0.009], [-0.074, -0.092]], B = [[-0.046, 0.022, -0.101], [-0.021, -0.016, 0.054]], alpha = 16` → `([[-0.7182, -1.2898, -0.5503], [0.0588, -2.3463, -0.1507], [-1.2846, -0.7153, -0.6088], [-0.2773, 0.4088, 1.06]], 0.0)`

In [ ]:
def merge(x, W, A, B, alpha):
    """
    After training, LoRA can be folded into the weights so serving has no extra
    cost. Return (merged W rounded to 4, max |difference| between x·W' and the
    LoRA forward, rounded to 8).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(merge, [
    (([[0.13, -0.13, 0.64, 0.1], [-0.54, 0.36, 1.3, 0.95]], [[-0.7, -1.27, -0.62], [0.04, -2.33, -0.22], [-1.25, -0.73, -0.54], [-0.32, 0.41, 1.04]], [[-0.013, 0.137], [-0.067, 0.035], [0.09, 0.009], [-0.074, -0.092]], [[-0.046, 0.022, -0.101], [-0.021, -0.016, 0.054]], 16), ([[-0.7182, -1.2898, -0.5503], [0.0588, -2.3463, -0.1507], [-1.2846, -0.7153, -0.6088], [-0.2773, 0.4088, 1.06]], 0.0)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Nested loops to build W', then compare outputs.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

NumPy outer product of A and B.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Nested loops to build W', then compare outputs.

```python
def merge(x, W, A, B, alpha):
    r = len(A[0])
    merged = [[W[i][j] + alpha / r * sum(A[i][t] * B[t][j] for t in range(r)) for j in range(len(W[0]))] for i in range(len(W))]
    out_merged = [[sum(x[n][i] * merged[i][j] for i in range(len(W))) for j in range(len(W[0]))] for n in range(len(x))]
    out_lora = np.array(x) @ np.array(W) + alpha / r * (np.array(x) @ np.array(A)) @ np.array(B)
    diff = max(abs(out_merged[n][j] - out_lora[n][j]) for n in range(len(x)) for j in range(len(W[0])))
    return ([[round(v, 4) for v in row] for row in merged], round(float(diff), 8))
```

**Cost:** O(d·k·r) time, O(d·k) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** NumPy outer product of A and B.

```python
def merge(x, W, A, B, alpha):
    x, W, A, B = map(np.array, (x, W, A, B))
    merged = W + alpha / A.shape[1] * A @ B
    diff = np.abs(x @ merged - (x @ W + alpha / A.shape[1] * (x @ A) @ B)).max()
    return (np.round(merged, 4).tolist(), round(float(diff), 8))
```

**Cost:** O(d·k·r) time, O(d·k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Nested loops to build W', then compare outputs. That costs O(d·k·r) time and O(d·k) space."

"I can do better: NumPy outer product of A and B. That's O(d·k·r) time and O(d·k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/fine-tuning/merge-the-adapter-for-serving)

<a id="q13"></a>
### Q13. Will Training Fit on the GPU?

🟡 Interview standard · *Full vs LoRA vs QLoRA memory*

Rough training memory in GB for a `params_b`-billion model (activations ignored): **full** = weights bf16 (2 bytes) + gradients bf16 (2) + Adam states fp32 (8) + fp32 master weights (4) per param; **LoRA** = frozen bf16 weights (2/param) + 16 bytes per trainable LoRA param; **QLoRA** = 4-bit frozen weights (0.5/param) + 16 bytes per LoRA param. Return the three, rounded to 1.

**Examples**

- `params_b = 8, lora_params_n = 40000000` → `(128, 16.6, 4.6)`
- `params_b = 70, lora_params_n = 200000000` → `(1120, 143.2, 38.2)`

In [ ]:
def train_memory(params_b, lora_params_n):
    """
    Rough training memory in GB for a params_b-billion model (activations
    ignored): full = weights bf16 (2 bytes) + gradients bf16 (2) + Adam states
    fp32 (8) + fp32 master weights (4) per param; LoRA = frozen bf16 weights
    (2/param) + 16 bytes per trainable LoRA param; QLoRA = 4-bit frozen weights
    (0.5/param) + 16 bytes per LoRA param. Return the three, rounded to 1.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(train_memory, [
    ((8, 40000000), (128, 16.6, 4.6)),
    ((70, 200000000), (1120, 143.2, 38.2)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Add the parts step by step.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Three expressions.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Add the parts step by step.

```python
def train_memory(params_b, lora_params_n):
    p = params_b * 1e9
    full = (p * 2 + p * 2 + p * 8 + p * 4) / 1e9
    lora = (p * 2 + lora_params_n * 16) / 1e9
    qlora = (p * 0.5 + lora_params_n * 16) / 1e9
    return (round(full, 1), round(lora, 1), round(qlora, 1))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Three expressions.

```python
def train_memory(params_b, lora_params_n):
    extra = lora_params_n * 16 / 1e9
    return (round(params_b * 16, 1), round(params_b * 2 + extra, 1), round(params_b * 0.5 + extra, 1))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Add the parts step by step. That costs O(1) time and O(1) space."

"I can do better: Three expressions. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/fine-tuning/will-training-fit-on-the-gpu)

<a id="q14"></a>
### Q14. Only Learn From the Assistant's Words

🟡 Interview standard · *Loss masking*

In supervised fine-tuning, the loss is usually computed only on assistant tokens — the model shouldn't learn to imitate the user. Given per-token losses and each token's role, return `(mean loss over all tokens, mean over assistant tokens)`, rounded to 4.

**Examples**

- `losses = [2.1, 1.9, 2.4, 0.6, 0.4, 0.5, 3.0, 0.7], roles = ['user', 'user', 'user', 'assistant', 'assistant', 'assistant', 'user', 'assistant']` → `(1.45, 0.55)`

In [ ]:
def masked_loss(losses, roles):
    """
    In supervised fine-tuning, the loss is usually computed only on assistant
    tokens — the model shouldn't learn to imitate the user. Given per-token
    losses and each token's role, return (mean loss over all tokens, mean over
    assistant tokens), rounded to 4.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(masked_loss, [
    (([2.1, 1.9, 2.4, 0.6, 0.4, 0.5, 3.0, 0.7], ['user', 'user', 'user', 'assistant', 'assistant', 'assistant', 'user', 'assistant']), (1.45, 0.55)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop with a mask.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

NumPy boolean mask.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop with a mask.

```python
def masked_loss(losses, roles):
    total = n = 0
    for l, r in zip(losses, roles):
        if r == "assistant":
            total += l
            n += 1
    return (round(sum(losses) / len(losses), 4), round(total / n, 4))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** NumPy boolean mask.

```python
def masked_loss(losses, roles):
    l, mask = np.array(losses), np.array(roles) == "assistant"
    return (round(float(l.mean()), 4), round(float(l[mask].mean()), 4))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop with a mask. That costs O(n) time and O(1) space."

"I can do better: NumPy boolean mask. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/fine-tuning/only-learn-from-the-assistants-words)

<a id="q15"></a>
### Q15. Pack Short Examples Together

🟡 Interview standard · *Packing*

Training runs on fixed-length sequences of `max_len` tokens. Pack examples (in order) into sequences first-fit: each goes into the first sequence with room, else a new one. Return `(sequences used, % of tokens that are padding rounded to 1)`.

**Examples**

- `lengths = [900, 300, 1500, 200, 700, 1800, 100, 400], max_len = 2048` → `(4, 28.0)`

In [ ]:
def pack(lengths, max_len):
    """
    Training runs on fixed-length sequences of max_len tokens. Pack examples (in
    order) into sequences first-fit: each goes into the first sequence with
    room, else a new one. Return (sequences used, % of tokens that are padding
    rounded to 1).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(pack, [
    (([900, 300, 1500, 200, 700, 1800, 100, 400], 2048), (4, 28.0)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A list of remaining space per sequence, scanned for each example.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same first-fit with `next` over enumerated free space.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A list of remaining space per sequence, scanned for each example.

```python
def pack(lengths, max_len):
    free = []
    for L in lengths:
        placed = False
        for i in range(len(free)):
            if free[i] >= L:
                free[i] -= L
                placed = True
                break
        if not placed:
            free.append(max_len - L)
    return (len(free), round(sum(free) / (len(free) * max_len) * 100, 1))
```

**Cost:** O(n · s) time, O(s) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same first-fit with `next` over enumerated free space.

```python
def pack(lengths, max_len):
    free = []
    for L in lengths:
        i = next((i for i, f in enumerate(free) if f >= L), None)
        if i is None:
            free.append(max_len - L)
        else:
            free[i] -= L
    return (len(free), round(sum(free) / (len(free) * max_len) * 100, 1))
```

**Cost:** O(n · s) time, O(s) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A list of remaining space per sequence, scanned for each example. That costs O(n · s) time and O(s) space."

"I can do better: Same first-fit with `next` over enumerated free space. That's O(n · s) time and O(s) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `lengths = [900, 300, 1500, 200], max_len = 2048`

1. GeneratorExit raised: 
2. GeneratorExit raised: 
3. pack() returns (2, 29.2)

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/fine-tuning/pack-short-examples-together)

<a id="q16"></a>
### Q16. Clean the Training Set

🟡 Interview standard · *Data quality*

Drop examples that are (in this order of checks) missing a response ('empty'), have a prompt plus response longer than `max_chars` ('too long'), are exact duplicates of an earlier example ('duplicate'), or whose prompt appears in the eval set ('eval leak'). Return `(kept count, {reason: count})`.

**Examples**

- `examples = [('hi', 'hello'), ('2+2', ''), ('hi', 'hello'), ('long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long ', 'x'), ('capital of France', 'Paris'), ('refund?', '10 days')], eval_prompts = ['capital of France'], max_chars = 100` → `(2, {'empty': 1, 'duplicate': 1, 'too long': 1, 'eval leak': 1})`

In [ ]:
def clean_data(examples, eval_prompts, max_chars):
    """
    Drop examples that are (in this order of checks) missing a response
    ('empty'), have a prompt plus response longer than max_chars ('too long'),
    are exact duplicates of an earlier example ('duplicate'), or whose prompt
    appears in the eval set ('eval leak'). Return (kept count, {reason: count}).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(clean_data, [
    (([('hi', 'hello'), ('2+2', ''), ('hi', 'hello'), ('long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long long ', 'x'), ('capital of France', 'Paris'), ('refund?', '10 days')], ['capital of France'], 100), (2, {'empty': 1, 'duplicate': 1, 'too long': 1, 'eval leak': 1})),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

An if-chain with a seen set.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A rule list of (reason, test) checked in order.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** An if-chain with a seen set.

```python
def clean_data(examples, eval_prompts, max_chars):
    seen, kept, reasons = set(), 0, {}
    for p, r in examples:
        reason = None
        if not r.strip():
            reason = "empty"
        elif len(p) + len(r) > max_chars:
            reason = "too long"
        elif (p, r) in seen:
            reason = "duplicate"
        elif p in eval_prompts:
            reason = "eval leak"
        seen.add((p, r))
        if reason:
            reasons[reason] = reasons.get(reason, 0) + 1
        else:
            kept += 1
    return (kept, reasons)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A rule list of (reason, test) checked in order.

```python
def clean_data(examples, eval_prompts, max_chars):
    from collections import Counter
    seen, kept, reasons, evals = set(), 0, Counter(), set(eval_prompts)
    for p, r in examples:
        rules = [("empty", not r.strip()), ("too long", len(p) + len(r) > max_chars),
                 ("duplicate", (p, r) in seen), ("eval leak", p in evals)]
        reason = next((name for name, hit in rules if hit), None)
        seen.add((p, r))
        if reason:
            reasons[reason] += 1
        else:
            kept += 1
    return (kept, dict(reasons))
```

**Cost:** O(n · r) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: An if-chain with a seen set. That costs O(n) time and O(n) space."

"I can do better: A rule list of (reason, test) checked in order. That's O(n · r) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `examples = [('hi', 'hello'), ('2+2', ''), ('hi', 'hello')], eval_prompts = [], max_chars = 100`

1. GeneratorExit raised: 
2. GeneratorExit raised: 
3. clean\_data() returns (1, {'empty': 1, 'duplicate': 1})

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/fine-tuning/clean-the-training-set)

<a id="q17"></a>
### Q17. The DPO Loss for One Pair

🟡 Interview standard · *Preference tuning*

Direct Preference Optimisation pushes the model to prefer the chosen answer over the rejected one *relative to a frozen reference model*. Given log-probabilities, return `(loss, implicit reward margin)` rounded to 4, where margin = β·[(lp\_chosen − ref\_chosen) − (lp\_rejected − ref\_rejected)] and loss = −log σ(margin).

**Examples**

- `lp_chosen = -12.0, lp_rejected = -15.0, ref_chosen = -13.0, ref_rejected = -14.0, beta = 0.1` → `(0.5981, 0.2)`
- `lp_chosen = -14.0, lp_rejected = -12.0, ref_chosen = -13.0, ref_rejected = -13.0, beta = 0.1` → `(0.7981, -0.2)`

In [ ]:
def dpo_loss(lp_chosen, lp_rejected, ref_chosen, ref_rejected, beta):
    """
    Direct Preference Optimisation pushes the model to prefer the chosen answer
    over the rejected one relative to a frozen reference model. Given log-
    probabilities, return (loss, implicit reward margin) rounded to 4, where
    margin = β·[(lp_chosen − ref_chosen) − (lp_rejected − ref_rejected)] and
    loss = −log σ(margin).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(dpo_loss, [
    ((-12.0, -15.0, -13.0, -14.0, 0.1), (0.5981, 0.2)),
    ((-14.0, -12.0, -13.0, -13.0, 0.1), (0.7981, -0.2)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute the margin, then the sigmoid by hand.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Use the stable form −log σ(m) = log(1 + e^(−m)) via `np.logaddexp(0, −m)`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Compute the margin, then the sigmoid by hand.

```python
def dpo_loss(lp_chosen, lp_rejected, ref_chosen, ref_rejected, beta):
    margin = beta * ((lp_chosen - ref_chosen) - (lp_rejected - ref_rejected))
    sigmoid = 1 / (1 + math.exp(-margin))
    return (round(-math.log(sigmoid), 4), round(margin, 4))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Use the stable form −log σ(m) = log(1 + e^(−m)) via `np.logaddexp(0, −m)`.

```python
def dpo_loss(lp_chosen, lp_rejected, ref_chosen, ref_rejected, beta):
    m = beta * ((lp_chosen - ref_chosen) - (lp_rejected - ref_rejected))
    return (round(float(np.logaddexp(0, -m)), 4), round(m, 4))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute the margin, then the sigmoid by hand. That costs O(1) time and O(1) space."

"I can do better: Use the stable form −log σ(m) = log(1 + e^(−m)) via `np.logaddexp(0, −m)`. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/fine-tuning/the-dpo-loss-for-one-pair)

<a id="q18"></a>
### Q18. A Batch of Preference Pairs

🟡 Interview standard · *DPO metrics*

For a batch of pairs `(lp_chosen, lp_rejected, ref_chosen, ref_rejected)` return `(mean DPO loss, reward accuracy)` rounded to 4, where reward accuracy is the share of pairs with a positive margin — the main number to watch during DPO training.

**Examples**

- `pairs = [(-12, -15, -13, -14), (-14, -12, -13, -13), (-10, -20, -11, -18), (-9, -9, -10, -10)], beta = 0.1` → `(0.6609, 0.5)`

In [ ]:
def dpo_batch(pairs, beta):
    """
    For a batch of pairs (lp_chosen, lp_rejected, ref_chosen, ref_rejected)
    return (mean DPO loss, reward accuracy) rounded to 4, where reward accuracy
    is the share of pairs with a positive margin — the main number to watch
    during DPO training.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(dpo_batch, [
    (([(-12, -15, -13, -14), (-14, -12, -13, -13), (-10, -20, -11, -18), (-9, -9, -10, -10)], 0.1), (0.6609, 0.5)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over pairs.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

NumPy arrays for the whole batch.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop over pairs.

```python
def dpo_batch(pairs, beta):
    total, correct = 0.0, 0
    for c, r, rc, rr in pairs:
        m = beta * ((c - rc) - (r - rr))
        total += math.log(1 + math.exp(-m))
        if m > 0:
            correct += 1
    return (round(total / len(pairs), 4), round(correct / len(pairs), 4))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** NumPy arrays for the whole batch.

```python
def dpo_batch(pairs, beta):
    c, r, rc, rr = np.array(pairs, dtype=float).T
    m = beta * ((c - rc) - (r - rr))
    return (round(float(np.logaddexp(0, -m).mean()), 4), round(float((m > 0).mean()), 4))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over pairs. That costs O(n) time and O(1) space."

"I can do better: NumPy arrays for the whole batch. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/fine-tuning/a-batch-of-preference-pairs)

<a id="q19"></a>
### Q19. Build Preference Pairs From Ratings

🟡 Interview standard · *Preference data*

Each prompt has several rated responses. Make one pair per prompt — the best-rated as 'chosen', the worst as 'rejected' — but only if the rating gap is at least `min_gap` (small gaps are noise). Return `[(prompt, chosen, rejected)]`.

**Examples**

- `ratings = {'p1': [('a', 4.5), ('b', 2.0), ('c', 3.9)], 'p2': [('d', 3.0), ('e', 3.2)], 'p3': [('f', 5.0), ('g', 1.0)]}, min_gap = 1.0` → `[('p1', 'a', 'b'), ('p3', 'f', 'g')]`

In [ ]:
def make_pairs(ratings, min_gap):
    """
    Each prompt has several rated responses. Make one pair per prompt — the
    best-rated as 'chosen', the worst as 'rejected' — but only if the rating gap
    is at least min_gap (small gaps are noise). Return [(prompt, chosen,
    rejected)].
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(make_pairs, [
    (({'p1': [('a', 4.5), ('b', 2.0), ('c', 3.9)], 'p2': [('d', 3.0), ('e', 3.2)], 'p3': [('f', 5.0), ('g', 1.0)]}, 1.0), [('p1', 'a', 'b'), ('p3', 'f', 'g')]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Find max and min by looping.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`max` and `min` with a key.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Find max and min by looping.

```python
def make_pairs(ratings, min_gap):
    out = []
    for prompt, responses in ratings.items():
        best = worst = responses[0]
        for resp in responses:
            if resp[1] > best[1]:
                best = resp
            if resp[1] < worst[1]:
                worst = resp
        if best[1] - worst[1] >= min_gap:
            out.append((prompt, best[0], worst[0]))
    return out
```

**Cost:** O(n) time, O(p) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `max` and `min` with a key.

```python
def make_pairs(ratings, min_gap):
    out = []
    for prompt, rs in ratings.items():
        best, worst = max(rs, key=lambda r: r[1]), min(rs, key=lambda r: r[1])
        if best[1] - worst[1] >= min_gap:
            out.append((prompt, best[0], worst[0]))
    return out
```

**Cost:** O(n) time, O(p) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Find max and min by looping. That costs O(n) time and O(p) space."

"I can do better: `max` and `min` with a key. That's O(n) time and O(p) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/fine-tuning/build-preference-pairs-from-ratings)

<a id="q20"></a>
### Q20. Prompt, RAG or Fine-Tune?

🟡 Interview standard · *Choosing the technique*

Recommend an approach from a problem's traits, in this order: needs fresh or private facts → 'RAG'; needs a new skill, format or style learned from many examples and you have ≥ 500 examples → 'fine-tune'; needs lower cost/latency at high volume and you have ≥ 500 examples → 'fine-tune a smaller model (distil)'; otherwise 'prompt engineering'.

**Examples**

- `t = {'fresh_facts': True, 'examples': 5000}` → `'RAG'`
- `t = {'new_format': True, 'examples': 2000}` → `'fine-tune'`
- `t = {'new_format': True, 'examples': 50}` → `'prompt engineering'`

In [ ]:
def choose_technique(t):
    """
    Recommend an approach from a problem's traits, in this order: needs fresh or
    private facts → 'RAG'; needs a new skill, format or style learned from many
    examples and you have ≥ 500 examples → 'fine-tune'; needs lower cost/latency
    at high volume and you have ≥ 500 examples → 'fine-tune a smaller model
    (distil)'; otherwise 'prompt engineering'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(choose_technique, [
    (({'fresh_facts': True, 'examples': 5000},), 'RAG'),
    (({'new_format': True, 'examples': 2000},), 'fine-tune'),
    (({'new_format': True, 'examples': 50},), 'prompt engineering'),
    (({'cost_pressure': True, 'examples': 20000},), 'fine-tune a smaller model (distil)'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

An if-chain.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

An ordered rule list.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** An if-chain.

```python
def choose_technique(t):
    many = t.get("examples", 0) >= 500
    if t.get("fresh_facts"):
        return "RAG"
    if t.get("new_format") and many:
        return "fine-tune"
    if t.get("cost_pressure") and many:
        return "fine-tune a smaller model (distil)"
    return "prompt engineering"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** An ordered rule list.

```python
def choose_technique(t):
    many = t.get("examples", 0) >= 500
    rules = [(t.get("fresh_facts"), "RAG"), (t.get("new_format") and many, "fine-tune"),
             (t.get("cost_pressure") and many, "fine-tune a smaller model (distil)")]
    return next((name for ok, name in rules if ok), "prompt engineering")
```

**Cost:** O(r) time, O(r) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: An if-chain. That costs O(1) time and O(1) space."

"I can do better: An ordered rule list. That's O(r) time and O(r) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/fine-tuning/prompt-rag-or-fine-tune)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. Why does LoRA work with so few parameters?</b></summary>

The weight changes needed to adapt a pre-trained model to a task have low intrinsic rank, so a product of two thin matrices captures most of the useful update.

</details>

<details>
<summary><b>2. Fine-tuning or RAG for company knowledge?</b></summary>

RAG. Knowledge changes and must be cited and kept current; fine-tuning bakes in behaviour, not reliable, updatable facts. Fine-tune for format, style or skills — sometimes on top of RAG.

</details>

<details>
<summary><b>3. What is QLoRA?</b></summary>

Training LoRA adapters on top of a base model stored in 4-bit precision, cutting memory roughly fourfold so large models can be tuned on one GPU.

</details>

<details>
<summary><b>4. Why mask the loss on user tokens?</b></summary>

So the model learns to produce assistant responses, not to predict or imitate user messages, and so gradients focus on the behaviour you want.

</details>

<details>
<summary><b>5. How does DPO differ from RLHF?</b></summary>

RLHF trains a reward model and optimises the policy with reinforcement learning (e.g. PPO). DPO optimises the policy directly on preference pairs with a closed-form loss relative to a reference model — simpler and more stable.

</details>

<details>
<summary><b>6. How do you know a fine-tune helped?</b></summary>

Compare against a well-prompted baseline on a held-out eval set for the target task, and re-check general and safety benchmarks for regressions.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: When fine-tuning is the right tool
- [ ] I can explain: LoRA: train a tiny update instead of the whole model
- [ ] I can explain: Memory: full vs LoRA vs QLoRA
- [ ] I can explain: Data and training: where results are won
- [ ] I can explain: Preference tuning with DPO

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Clean the Training Set
- [ ] [Q17](#q17) The DPO Loss for One Pair
- [ ] [Q18](#q18) A Batch of Preference Pairs
- [ ] [Q19](#q19) Build Preference Pairs From Ratings
- [ ] [Q20](#q20) Prompt, RAG or Fine-Tune?

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.